# 3 · FIR II filtering

**Local Quartus workflow · host or container · one active FPGA design at a time**

`y[n]=Σ h[k]x[n−k]`

**What is added:** A selectable, reloadable 16-tap FIR II filter.

**Why it is useful:** Compare noise/interference reduction with the original capture and see what filtering removes.

**How it works:** Vendor FIR II multiplies samples by Q15 coefficients and sums the taps. FPGA snapshot logic returns paired raw/filtered data with defined rounding and saturation.

```text
Captured IQ → FIR II → raw / filtered comparison
```

Python model → native VDS diagram → real RTL/IP simulation → SOF → board → focused CyberEther page.

## Setup

Once, from this example folder: `bash scripts/install_local.sh --system` (Linux system packages need sudo), then select **Atum A3 IQ Demos** as the notebook kernel. Quartus/Questa/Nios tools must already be installed. Set the optional paths below if they are not on PATH. No Docker commands are needed. Long-running cells show an animated activity bar, elapsed time and the active step/log; a full bar means successful completion, not an estimated compile percentage.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/lessons.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "cyberether"))
# Optional overrides, set before tool discovery:
# os.environ["QUARTUS_ROOTDIR"] = "/your/install/quartus"
# os.environ["QUESTA_ROOT"] = "/your/install/questa_fse"
# os.environ["ATUM_ALTERA_ROOT"] = "/your/install"
# os.environ["ATUM_UART"] = "/dev/serial/by-id/your-board"
# os.environ["ATUM_JTAG_CABLE"] = "your cable from jtagconfig"
from lessons import Lesson, lan_ip, stop_demo
from common import tool_exe
CHECKPOINT = 3
FORCE_REBUILD = False   # Fresh checkout builds; source-matched SOFs are reused afterwards.
lesson = Lesson(CHECKPOINT)
HOST = lan_ip()
print("Python:", sys.executable)
print("Quartus:", tool_exe("quartus_sh"))
print("Questa:", tool_exe("vsim"))
print("Browser host:", HOST)


## 1 · Python simulation

The code below is the editable numerical model. Its plots are synthetic/theory, separate from the RTL results and live hardware below.

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    import numpy as np
    import matplotlib.pyplot as plt
    from notebook_plots import spectrum
    from nios_fir import PROFILES, fixed_reference
    from nios_ddc import reference, gain_reference
    from iq_lab_samples import load_sample
    raw, _ = load_sample("two-tone")
    fs = 16e6
    n = CHECKPOINT
    coeff = PROFILES[2]
    out = fixed_reference(raw, coeff, True)
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    axs[0].stem(np.arange(16), np.array(coeff) / 32768)
    axs[0].set(xlabel='Tap', ylabel='Q15 / 32768', title='Actual reloadable Hamming coefficients')
    axs[1].plot(raw[:160, 0], label='Before I')
    axs[1].plot(out[:160, 0], label='After I')
    axs[1].set(xlabel='Sample', ylabel='CI8 count', title='Integer convolution + rounding')
    axs[1].legend()
    spectrum(axs[2], raw.astype(float), fs, 'Input')
    spectrum(axs[2], out.astype(float), fs, 'FIR output')
    summary = {'taps': 16, 'reference': 'Python fixed-point convolution; nearest rounding and saturation'}
    fig.suptitle(f"{CHECKPOINT} · Python simulation / integer reference")
    fig.tight_layout()
    plt.show()
    summary

summary

### Filter library · preview without hardware

There are **15 examples** stored in Python and one active 16-tap coefficient set in the FPGA. The same real coefficients filter I and Q. Selecting another example reloads the coefficients through Nios V; it does not rebuild the bitfile.

Try **narrow low-pass**, **high-pass**, **band-pass**, or the **4 MHz notch** with the FPGA two-tone input. The short filters have broad transition regions. Frequencies scale with sample rate: the matching FIR test notch is at 0.25 Fs, so it is 4 MHz only at 16 MS/s.

Q15 represents each coefficient divided by 32768. All examples contain exactly 16 integers; shorter designs have trailing zeros. The example's L1 gain bounds worst-case amplification, so some arbitrary inputs can clip even when a sinusoidal passband is near unity. [Filter guide](../docs/FIR_FILTER_LIBRARY.md).

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    from IPython.display import Markdown, display
    from fir_filters import catalog, plot_filter, q15
    rows = ["| Python name | Example | What to observe |", "|---|---|---|"]
    rows += [f"| `{f['id']}` | {f['title']} | {f['description']} |" for f in catalog()]
    display(Markdown("\n".join(rows)))

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    from fir_filters import plot_catalog
    plot_catalog(rate=16_000_000)

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    FILTER = "notch-4mhz"  # Change to any Python name in the table.
    preview = plot_filter(FILTER, rate=16_000_000)
    print(preview)

## 2 · Generate the actual FPGA block design

Rebuild the Nios initialization memory where present, recreate the native VDS system, validate and generate vendor IP. The cell opens **Quartus Visual Designer Studio itself**, fits the diagram and uses **File → Export**. Its native PDF is rendered to a high-resolution PNG and displayed inline. The original PDF and a full GUI screenshot are saved beside it. A private X display keeps capture separate from your desktop. The output also provides **Quartus project** and **complete project ZIP** links. The ZIP contains the SDC, RTL, generated IP and Nios memory initialization. Extract it before opening the QPF in Quartus.

Timing constraints: [iq_bridge_vds.sdc](../fpga_hub/iq_bridge/vds_demo_03_fir/iq_bridge_vds.sdc).

In [ ]:
from IPython.display import Image, display
diagram_png = lesson.generate()
display(Image(filename=str(diagram_png)))
print("Exported:", diagram_png)

### Saved native VDS GUI diagram

![Checkpoint 3 — actual VDS GUI export](diagrams/03_vds_gui.png)

The cell above refreshes this diagram from your local design.

### Official Nios V BSP and applications

Generate a HAL BSP from this actual VDS system, plus editable **Hello World** and **IQ monitor** CMake applications. This builds software only; it does not download or replace the running firmware. Open both BSP and app with **RiscFree → File → Import Nios V CMake Project**, then build with Ctrl+B. See [Nios V IDE guide](../software/README.md) for BSP Editor, JTAG download and restore steps.

In [ ]:
software_dir = lesson.software()
print("BSP:", software_dir / "bsp")
print("Hello World:", software_dir / "hello_world")
print("IQ monitor:", software_dir / "iq_monitor")

## 3 · RTL/IP simulation

Actual generated FIR II VHDL plus capture controller; impulse, custom coefficients, rounding and saturation.

The plots below read the actual simulator output files. Assertions must pass before the result is plotted.

In [ ]:
simulation_result = lesson.simulate()
print("Simulation:", simulation_result["status"])
lesson.simulation_plot()

### Open the Questa GUI

Use a desktop terminal **on the machine running Quartus** (or its container desktop). Execute the printed commands. `wave.wlf` opens the saved real simulation; `view.do` adds its signals. To rerun interactively, start `vsim -gui` in the same directory and type `do gui.do` in the Transcript. This reruns the same assertions and leaves the waveform window open.

In [ ]:
lesson.questa_gui()

## 4 · Build the SOF

A source-matched, hash-verified cached bitfile can be reused for a quick presentation. Set `FORCE_REBUILD=True` to demonstrate a full Quartus compilation. A fresh checkout always requires a build. Both a new build and cached reuse display links to the complete Quartus project and its ZIP. Open the linked QPF (or extract the ZIP first), then use **Processing → Start Compilation** to build in the GUI.

In [ ]:
build = lesson.build(force=FORCE_REBUILD)
print("SOF:", lesson.stage / "iq_bridge_vds.sof")
print("SHA256:", build["sha256"])
print("Compiled resources:", build["resources"])

## 5 · Program the physical board

This cell **programs the selected checkpoint**. It first stops a previous notebook-owned demo and releases UART/JTAG. Stop any unrelated acquisition service before running it; an unknown serial owner is refused. No ESP32 flashing or reset-button presses are needed with the installed suite.

In [ ]:
lesson.program()

## 6 · Focused CyberEther demo

Press **FPGA impulse check**: the first sample should spread into the filter impulse response. Then try **FPGA two-tone check**, toggle Hamming/Boxcar, and compare the common-axis input/output spectrum. The page checks the actual pattern, coefficients, rounding, payload CRC and integer error. **Live RF** returns to the radio.

The page uses the bitfile just programmed. Diagnostic plots work without WebRTC; the native CyberEther canvas is expandable below them. **VERIFIED** requires a fresh matching result; **APPLYING**, **HELD**, **STALE** or **ERROR** keeps old data visibly separate from successful application. **Software preview** is host-only and never counts as a hardware pass. See [demo guide](../docs/NOTEBOOK_DEMOS.md). One board and one acquisition owner are shared across all notebooks.

In [ ]:
live = lesson.start_demo(host=HOST)
print(live)

### Load a filter through Python · keep CyberEther running

Run this after the launch cell. Python sends a request to the **existing acquisition owner**; it does not open a second UART/JTAG client. The call waits for a fresh FPGA result, verifies all coefficients, CRC/readback and zero integer-reference error, and reports clipping. The selected filter remains active across captures until you choose another filter or restart the demo. The shared progress row shows application/verification feedback.

In the browser choose an example, then **Apply selected example**. Open **Filter design** to preview the candidate or edit all 16 Q15 integers. Previewing or editing alone never changes the FPGA. Optional: embed the same page below; its output contains a private invite, just like the launch cell.

In [ ]:
from fir_controls import FIRControls
fir = FIRControls()
result = fir.apply(FILTER, input_kind="fpga-tones")
print(result)

#### Your own coefficients

Edit the 16 integers below, then run the cell. This eight-sample average uses eight Q15 taps of 4096 and eight zeros. To design using real-valued taps, call `q15([0.25] * 4)`; the helper pads shorter filters and rejects overflow. Positive 1.0 is not representable in signed Q15; use 32767/32768 for a near-unity tap. `plot_filter(coefficients=CUSTOM_Q15)` previews your actual quantized design without changing hardware.

Change `input_kind` to `fpga-impulse` to inspect its impulse response, or `live` for real RF. Real RF uses the notebook's current radio/channel/rate. These reloads affect the 16-tap FIR, not the separate fixed 127-tap DDC filters.

In [ ]:
CUSTOM_Q15 = [4096] * 8 + [0] * 8
print(fir.apply(coefficients=CUSTOM_Q15, input_kind="fpga-impulse"))

In [ ]:
# Interactive GUI: the same acquisition owner and filter settings as the browser.
fir.show_gui(height=1050)

Leave the page running while presenting. The next notebook stops it before programming. To stop it manually, run `stop_demo()` in the cell below. Live invites and captures stay in ignored runtime folders; do not publish their tokens.

In [ ]:
# stop_demo()

### CyberEther · example screenshot

Saved browser screenshot from a physical Atum A3 Nano / Agilex 3 hardware test. This is a static example; run the launch cell above for current data.

FPGA-generated two-tone input with the 4 MHz notch applied. Compare the cyan input with the gold filtered output and inspect the coefficient readback evidence.

![CyberEther example — lesson 03](screenshots/03_cyberether.png)

[Open full-resolution screenshot](screenshots/03_cyberether.png).

#### Custom FIR editor example

Eight-tap averaging coefficients loaded through the browser editor. The candidate response, actual before/after samples and VERIFIED coefficient readback are shown together. The input is an FPGA-generated test signal.

![CyberEther custom FIR coefficient editor](screenshots/03_fir_custom_editor.png)

[Open full-resolution editor screenshot](screenshots/03_fir_custom_editor.png).